In [1]:
# Full executable script: Adaptive Frequency Weighting ViT (Final for NCC)
# Save as a .py and run, or paste into a notebook cell.
# Requirements: torch, torchvision, timm, numpy, pandas, scikit-learn, matplotlib, opencv-python, tqdm
# Optional: umap-learn

import os
import json
import math
import glob
import time
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data

import timm
from timm.models.vision_transformer import Attention  # for attention replacement

import torchvision.transforms as transforms
import torchvision.models as models

import cv2
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc

try:
    import umap  # optional
except Exception:
    umap = None

# -----------------------------
# SECTION 1: DATA LOADING
# -----------------------------

def np_load_frame(filename, resize_height, resize_width):
    """Loads and preprocesses a single image frame."""
    img = cv2.imread(filename)
    if img is None:
        print(f"[WARN] Could not read image: {filename}")
        return None
    img = cv2.resize(img, (resize_width, resize_height))
    # Convert BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 127.5 - 1.0 # Normalize to [-1, 1]
    return img

class AnomalyDataset(data.Dataset):
    """Dataset for loading sequences of frames for anomaly detection."""
    def __init__(self, video_folder, transform, resize_h, resize_w, time_step=4, num_pred=1, augment=False):
        self.dir = video_folder
        self.transform = transform
        self._H, self._W = resize_h, resize_w
        self._T, self._P = time_step, num_pred
        self.augment = augment

        # Define augmentation pipeline
        self.aug_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor()
        ])
        self.video_frames = []
        self.index_samples = []
        self.setup()

    def setup(self):
        """Initializes the list of frames and sample indices."""
        videos = glob.glob(os.path.join(self.dir, '*'))
        if not videos:
            print(f"[WARN] No files/folders found in {self.dir}")
            return
        all_frames = []
        # Check if the first item is a directory (dataset structure: videos/frames)
        if os.path.isdir(videos[0]):
            for v in sorted(videos):
                frames = sorted(glob.glob(os.path.join(v, '*.jpg')),
                                key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))
                all_frames.extend(frames)
        else: # Dataset structure: flat list of frames
            all_frames = sorted([f for f in videos if f.lower().endswith(('.png', '.jpg', 'jpeg'))],
                                key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))

        if not all_frames:
             print(f"[WARN] No image files found in {self.dir} or its subdirectories.")
             return

        self.video_frames = all_frames
        # Ensure enough frames for at least one sequence
        if len(all_frames) >= (self._T + self._P):
             self.index_samples = list(range(len(all_frames) - (self._T + self._P) + 1))
        else:
             print(f"[WARN] Not enough frames ({len(all_frames)}) in {self.dir} for sequence length {self._T + self._P}.")
             self.index_samples = []


    def __getitem__(self, index):
        start = self.index_samples[index]
        # Pre-allocate numpy array for efficiency
        arr = np.zeros((self._T + self._P, 3, self._H, self._W), dtype=np.float32)

        # Consistent augmentation for the sequence
        apply_aug = self.augment and (np.random.rand() > 0.5)

        for i in range(self._T + self._P):
            frame = np_load_frame(self.video_frames[start + i], self._H, self._W)
            if frame is None:
                # Handle missing frame - fill with zeros or previous frame? Zeros for now.
                print(f"[WARN] Filling missing frame {self.video_frames[start + i]} with zeros.")
                frame_tensor = torch.zeros(3, self._H, self._W)
            elif apply_aug:
                 # Convert numpy [-1,1] HWC to uint8 [0,255] HWC for PIL transforms
                frame_uint8 = ((frame + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
                # Apply augmentations (ToPILImage, Flip, Jitter, ToTensor) -> tensor [0,1] CHW
                aug_tensor = self.aug_transform(frame_uint8)
                # Convert back to [-1,1] CHW numpy
                frame_tensor = (aug_tensor * 2.0 - 1.0)
            else:
                # Apply only the basic transform (ToTensor equivalent for numpy)
                frame_tensor = self.transform(frame)

            arr[i] = frame_tensor
        return {'frames': torch.from_numpy(arr)}

    def __len__(self):
        return len(self.index_samples)

# -----------------------------
# SECTION 2: CONFIG
# -----------------------------

def get_config():
    """Returns a configuration object for the experiment."""
    class Cfg:
        exp_name = "AdaptiveFreqWeight_ViT_NCC"
        # Adjust paths for your environment (e.g., Kaggle, local)
        out_dir = "/kaggle/working/"
        train_folder = "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_train"
        test_folder  = "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_test"
        label_npy    = "/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy"

        train = True
        n_gpu = 1
        image_size = 224
        num_workers = 4 # Adjust based on your system cores/memory
        lr = 1e-4
        wd = 1e-3 # Slightly increased weight decay
        epochs = 25 # Increased epochs slightly
        batch_size = 8
        unfreeze_layers = 60 # Number of layers from the END of the model to unfreeze
        num_frames = 4 # Input sequence length

        # Baseline reconstruction loss weights (used when use_learnable_mask=False OR as part of combined loss)
        l1_weight = 0.15
        ssim_weight = 0.85
        perceptual_weight = 0.1 # Slightly increased perceptual weight

        # LR schedule
        warmup_steps = 100 # Steps for linear warmup
        eta_min = 1e-6     # Minimum LR for cosine annealing

        # --- Adaptive Frequency Weighting Novelty ---
        use_learnable_mask = True   # <<< Main toggle for the novelty
        frequency_guidance_weight = 0.2 #0.1,0.2, Weight for the frequency loss term guiding the MLP
        radial_mlp_hidden = 32      # 32,32, Hidden dim for the RadialMaskMLP
        radial_mlp_layers = 3     #2,3 Number of hidden layers for the RadialMaskMLP

        # Fixed frequency mask parameters (used if use_learnable_mask=False)
        low_freq_band_cutoff = 10
        high_freq_band_cutoff = 100

        # --- Analysis Toggles ---
        temporal_smoothing = True # Apply moving average to validation scores
        smoothing_window = 5      # Window size for smoothing (odd number recommended)
        do_fft_analysis = True
        do_attention_viz = True
        do_freq_score_pr = True   # Evaluate PR curve using frequency score

    return Cfg()

# -----------------------------
# SECTION 3: UTILS / METRICS
# -----------------------------

def setup_device(n_gpu_use):
    """Sets up the device for training."""
    n_gpu = torch.cuda.device_count()
    if n_gpu_use > 0 and n_gpu == 0:
        print("[WARN] No GPU available. Using CPU.")
        n_gpu_use = 0
    device = torch.device('cuda:0' if n_gpu_use > 0 else 'cpu')
    print(f"[INFO] Using device: {device}")
    return device, list(range(n_gpu_use))

class MetricTracker:
    """A simple class to track average values of metrics."""
    def __init__(self, *keys):
        self._data = pd.DataFrame(index=keys, columns=['total', 'counts', 'average'])
        self.reset()

    def reset(self):
        for c in self._data.columns:
            self._data[c].values[:] = 0

    def update(self, key, value, n=1):
        if key not in self._data.index:
             raise KeyError(f"Key '{key}' not found in MetricTracker")
        self._data.total[key] += value * n
        self._data.counts[key] += n
        self._data.average[key] = self._data.total[key] / self._data.counts[key]

    def result(self):
        return dict(self._data.average)

def create_frequency_band_mask(shape, low_cutoff, high_cutoff, device):
    """Creates a fixed mask to weight frequency bands."""
    # shape: (B, C, H, W)
    _, _, H, W = shape
    cy, cx = H // 2, W // 2
    # Create radial distance grid
    yy, xx = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing='ij')
    dist = torch.sqrt((xx - cx).float()**2 + (yy - cy).float()**2)
    # Define frequency bands
    low = (dist <= low_cutoff).float()
    high = (dist > high_cutoff).float()
    mid = 1.0 - low - high # Mid band is between low and high cutoffs
    # Assign weights (Example: higher weight for mid-band)
    weights = (low * 0.1) + (mid * 1.5) + (high * 0.2)
    # Return shape (1, 1, H, W) for broadcasting
    return weights.unsqueeze(0).unsqueeze(0)

# -----------------------------
# SECTION 4: LOSSES & MODELS
# -----------------------------

def create_window(window_size, channel, device):
    """Creates a 2D Gaussian window for SSIM."""
    _1D = torch.exp(-(torch.arange(window_size, dtype=torch.float, device=device) - window_size // 2)**2 / (2 * 1.5**2))
    _1D = _1D / _1D.sum()
    _2D = _1D[:, None] @ _1D[None, :]
    return _2D.expand(channel, 1, window_size, window_size).contiguous()

class SSIM(nn.Module):
    """Structural Similarity Index Measure."""
    def __init__(self, window_size=11, size_average=True):
        super().__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = -1 # Initialize channel
        self.window = None

    def forward(self, img1, img2):
        (_, ch, _, _) = img1.size()
        # Create window if dimensions/device change
        if self.window is None or ch != self.channel or self.window.device != img1.device:
            self.window = create_window(self.window_size, ch, img1.device)
            self.channel = ch

        pad = self.window_size // 2
        # Calculate mean, variance, covariance using convolution with Gaussian window
        mu1 = F.conv2d(img1, self.window, padding=pad, groups=ch)
        mu2 = F.conv2d(img2, self.window, padding=pad, groups=ch)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = F.conv2d(img1 * img1, self.window, padding=pad, groups=ch) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, self.window, padding=pad, groups=ch) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, self.window, padding=pad, groups=ch) - mu1_mu2
        # SSIM constants
        C1, C2 = 0.01**2, 0.03**2
        # SSIM formula
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

        if self.size_average:
            return ssim_map.mean()
        # Return per-sample mean SSIM if not averaging across batch
        return ssim_map.mean([1, 2, 3])

class PerceptualLoss(nn.Module):
    """VGG-based perceptual loss."""
    def __init__(self):
        super().__init__()
        # Use updated weights API
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        # Extract features up to conv4_4 (layer 35)
        self.slices = nn.ModuleList([vgg[i] for i in range(36)])
        # Freeze VGG parameters
        for p in self.parameters():
            p.requires_grad = False
        # Register normalization buffers (ImageNet mean/std)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        # Normalize inputs using ImageNet stats before feeding to VGG
        x = (x - self.mean) / self.std
        y = (y - self.mean) / self.std
        loss = 0.0
        # VGG layers used for feature comparison (relu1_2, relu2_2, relu3_4, relu4_4, relu5_4)
        feature_layers = {3, 8, 17, 26, 35}
        for i, layer in enumerate(self.slices):
            x = layer(x)
            y = layer(y)
            if i in feature_layers:
                loss += F.l1_loss(x, y) # L1 distance between features
        return loss

class RadialMaskMLP(nn.Module):
    """Small MLP to learn radial frequency weights."""
    def __init__(self, hidden_dim=32, layers=2):
        super().__init__()
        seq = []
        in_dim = 1 # Input is normalized radius (scalar)
        for _ in range(layers):
            seq.append(nn.Linear(in_dim, hidden_dim))
            seq.append(nn.ReLU(inplace=True))
            in_dim = hidden_dim
        seq.append(nn.Linear(in_dim, 1)) # Output is a single weight value
        self.net = nn.Sequential(*seq)

    def forward(self, radius_norm):
        # radius_norm is shape (H, W) or (B, 1, H, W)
        original_shape = radius_norm.shape
        # Flatten input for MLP
        flat_input = radius_norm.reshape(-1, 1)
        # Pass through MLP
        output_weights = self.net(flat_input)
        # Ensure weights are positive using Softplus
        output_weights = F.softplus(output_weights)
        # Reshape back to original spatial dimensions
        output_weights = output_weights.reshape(original_shape)
        # Normalize weights to range approx [0, 1] for stability
        output_weights = output_weights / (output_weights.max() + 1e-8)
        return output_weights

class AttentionWithMap(Attention):
    """Wrapper for timm's Attention module to store the attention map."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_map = None

    # MODIFIED: Added attn_mask=None to the signature
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        # Calculate QKV
        qkv = self.qkv(x).reshape(
            B, N, 3, self.num_heads, C // self.num_heads
        ).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0) # Make torchscript happy

        # Calculate attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale

        # ADDED: Check if mask is provided (though timm usually handles this before softmax)
        # It's good practice to acknowledge the argument even if not used here.
        if attn_mask is not None:
             # Depending on timm version/implementation, mask might be applied here or before.
             # For safety, let's assume it's handled before softmax by timm's Block,
             # but we acknowledge the argument. If errors persist, this part might need adjustment.
             # print("Warning: attn_mask received but not explicitly applied in AttentionWithMap.")
             pass # Assuming timm's Block handles applying the mask before softmax

        attn = attn.softmax(dim=-1)
        # Store the map AFTER softmax
        self.attn_map = attn

        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        # Apply output projection
        x = self.proj(x)
        return x

    def get_attention_map(self):
        """Returns the stored attention map."""
        return self.attn_map

class VisionTransformerSOTA(nn.Module):
    """Main ViT model with modified attention and optional RadialMaskMLP."""
    def __init__(self, image_size=224, num_frames=4, unfreeze_layers=60, cfg=None):
        super().__init__()
        # Load pretrained ViT-Base patch 16
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=True)
        emb_dim = self.encoder.head.in_features # Embedding dimension (768 for ViT-Base)

        # Replace original Attention modules with AttentionWithMap
        for blk in self.encoder.blocks:
            # Need to pass correct parameters from the original module
            new_attn = AttentionWithMap(
                dim=emb_dim,
                num_heads=blk.attn.num_heads,
                qkv_bias=True, # ViT-Base uses bias
                # Keep dropout rates if needed, though often turned off for eval
                # attn_drop=blk.attn.attn_drop.p,
                # proj_drop=blk.attn.proj_drop.p
                )
            # Load state dict to transfer pretrained weights
            new_attn.load_state_dict(blk.attn.state_dict())
            blk.attn = new_attn

        # Freeze early layers based on unfreeze_layers config
        if unfreeze_layers >= 0: # If -1, unfreeze all
             num_total_params = len(list(self.encoder.parameters()))
             num_to_freeze = num_total_params - unfreeze_layers
             if num_to_freeze > 0:
                  print(f"[INFO] Freezing {num_to_freeze} / {num_total_params} parameter groups in the encoder.")
                  for i, p in enumerate(self.encoder.parameters()):
                       if i < num_to_freeze:
                            p.requires_grad = False
        else:
            print("[INFO] Unfreezing all encoder parameters.")


        # Decoder network (upsamples concatenated features)
        dec_in_channels = emb_dim * num_frames # Input channels = features_per_frame * num_frames
        self.decoder = nn.Sequential(
            nn.Conv2d(dec_in_channels, 512, kernel_size=1),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh() # Output in [-1, 1] range
        )

        # Initialize RadialMaskMLP if configured
        self.use_learnable_mask = cfg is not None and getattr(cfg, "use_learnable_mask", False)
        if self.use_learnable_mask:
            print("[INFO] Using Learnable Radial Mask MLP.")
            self.radial_mlp = RadialMaskMLP(
                hidden_dim=getattr(cfg, "radial_mlp_hidden", 32),
                layers=getattr(cfg, "radial_mlp_layers", 2)
            )
        else:
            self.radial_mlp = None

    def forward(self, x):
        """Forward pass for training/prediction."""
        B, T, C, H, W = x.shape
        fmap_size = H // self.encoder.patch_embed.patch_size[0] # e.g., 224 // 16 = 14
        feats = []
        for i in range(T):
             # Get features from encoder, excluding CLS token
            f = self.encoder.forward_features(x[:, i])[:, 1:] # Output shape (B, N_patches, D)
            # Reshape features into a 2D map: (B, D, H_feat, W_feat)
            f = f.permute(0, 2, 1).reshape(B, -1, fmap_size, fmap_size)
            feats.append(f)
        # Concatenate features along the channel dimension
        z = torch.cat(feats, dim=1)
        # Pass concatenated features through the decoder
        return self.decoder(z)

    def forward_features(self, x_single_frame):
        """Helper to get features/attention for a single frame (for analysis)."""
        # Ensure input is single frame (B, C, H, W)
        assert x_single_frame.dim() == 4
        return self.encoder.forward_features(x_single_frame)

# -----------------------------
# SECTION 5: ANALYSIS & VIZ
# -----------------------------

def save_roc_curves(history, best_epoch_data, out_dir):
    """Saves ROC curve plots: one showing all epochs, one for the best epoch."""
    if not history:
        print("[WARN] ROC history is empty, cannot save ROC curves.")
        return

    # --- Plot ROC curves for all epochs ---
    plt.figure(figsize=(12, 9))
    # Generate colors for each epoch
    colors = plt.cm.viridis(np.linspace(0, 1, len(history)))
    for i, ep_data in enumerate(history):
        # Check if necessary data exists for the epoch
        if 'fpr' in ep_data and 'tpr' in ep_data and 'auc' in ep_data and ep_data['fpr'] is not None and ep_data['tpr'] is not None:
             # Ensure fpr and tpr are numpy arrays
            fpr = np.array(ep_data['fpr'])
            tpr = np.array(ep_data['tpr'])
            plt.plot(fpr, tpr, lw=1.5, color=colors[i],
                     label=f"Epoch {ep_data['epoch']} (AUC={ep_data['auc']:.4f})")
        else:
            print(f"[WARN] Missing ROC data for epoch {ep_data.get('epoch', 'N/A')}")

    plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance (AUC=0.5)') # Add chance line
    plt.title('ROC Curve Progression Over Epochs')
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.legend(loc='lower right') # Place legend appropriately
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.0])
    roc_all_path = os.path.join(out_dir, 'all_epochs_roc_curve.jpg')
    try:
        plt.savefig(roc_all_path, dpi=150) # Use reasonable DPI
        print(f"[INFO] Saved all epochs ROC curve to {roc_all_path}")
    except Exception as e:
        print(f"[ERROR] Failed to save all epochs ROC curve: {e}")
    plt.close() # Close the figure to free memory

    # --- Plot ROC curve for the best epoch only (Publication Style) ---
    if best_epoch_data and 'fpr' in best_epoch_data and 'tpr' in best_epoch_data and best_epoch_data['fpr'] is not None and best_epoch_data['tpr'] is not None:
        plt.figure(figsize=(6, 6))
        fpr = np.array(best_epoch_data['fpr'])
        tpr = np.array(best_epoch_data['tpr'])
        best_auc_val = best_epoch_data.get('auc', 0.0) # Get AUC, default to 0.0 if missing
        plt.plot(fpr, tpr, lw=2.5, color='red', # Use a distinct color
                 label=f"Proposed Method (AUC={best_auc_val:.4f})")
        plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance')
        plt.title('ROC Curve (Best Epoch)')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.legend(loc='lower right')
        plt.grid(True, linestyle=':', alpha=0.5) # Lighter grid for publication style
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.0])
        plt.gca().set_aspect('equal', adjustable='box') # Make axes equal
        roc_best_path = os.path.join(out_dir, 'publication_style_roc_curve.jpg')
        try:
            plt.savefig(roc_best_path, dpi=300) # Higher DPI for publication
            print(f"[INFO] Saved best epoch ROC curve to {roc_best_path}")
        except Exception as e:
            print(f"[ERROR] Failed to save best epoch ROC curve: {e}")
        plt.close() # Close the figure
    else:
        print("[WARN] Best epoch data missing or incomplete, cannot save publication style ROC curve.")

def save_anomaly_visualization(gt_frame, pred_frame, epoch, batch_idx, out_dir):
    """Saves a comparison image: Ground Truth | Prediction | Residual Map."""
    # Ensure tensors are on CPU and convert to numpy [0, 255] HWC uint8
    gt = (gt_frame.squeeze(0).permute(1, 2, 0).detach().cpu().numpy() + 1.0) * 127.5
    pr = (pred_frame.squeeze(0).permute(1, 2, 0).detach().cpu().numpy() + 1.0) * 127.5
    gt, pr = gt.astype(np.uint8), pr.astype(np.uint8)
    # Calculate absolute difference and apply colormap
    res_gray = cv2.cvtColor(cv2.absdiff(gt, pr), cv2.COLOR_RGB2GRAY)
    res_color = cv2.applyColorMap(res_gray, cv2.COLORMAP_JET)
    # Concatenate images horizontally
    combo = np.concatenate((gt, pr, res_color), axis=1)
    # Save the combined image
    save_path = os.path.join(out_dir, f"epoch_{epoch}_frame_{batch_idx}.jpg")
    cv2.imwrite(save_path, cv2.cvtColor(combo, cv2.COLOR_RGB2BGR)) # Convert back to BGR for OpenCV saving

# ... (Keep other analysis functions: save_roc_curves, analyze_frequency_domain, plot_radial_profile, analyze_attention_maps) ...
# IMPORTANT: Ensure analyze_attention_maps uses the modified ViT's get_attention_map correctly.

# --- Modified evaluate_frequency_score to use the learned mask ---
def evaluate_frequency_score(model, data_loader, device, out_dir, cfg):
    """Evaluates anomaly detection using the frequency score and plots PR curve."""
    print("\n[INFO] Evaluating Frequency Score Performance (PR Curve)...")
    model.eval()
    if not os.path.exists(cfg.label_npy):
        print("[WARN] label npy not found; skipping freq-score PR")
        return None # Return None if labels are missing
    labels_np = np.load(cfg.label_npy, allow_pickle=True)

    per_video_scores = []
    per_video_labels = []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(data_loader, desc="Freq Score Eval")):
            frames = batch['frames']
            inp = frames[:, :cfg.num_frames].to(device)
            tgt = frames[:, cfg.num_frames].to(device)
            try: # Get label for the target frame
                fn = int(os.path.basename(data_loader.dataset.video_frames[i + cfg.num_frames]).split('.')[0].split('_')[-1])
                y = labels_np[fn] if fn < len(labels_np) else 0
            except Exception as e:
                # print(f"Label extraction warning: {e}")
                y = 0

            pred = model(inp)
            residual = torch.abs(tgt - pred)
            residual_gray = torch.mean(residual, dim=1, keepdim=True) # Use grayscale for FFT
            resid_fft = torch.abs(torch.fft.fftshift(torch.fft.fft2(residual_gray, norm='ortho')))

            # --- Use Learned or Fixed mask based on config ---
            if cfg.use_learnable_mask and hasattr(model, "radial_mlp") and model.radial_mlp is not None:
                mask = radial_mask_from_mlp(model.radial_mlp, resid_fft.shape, device)
            else:
                mask = create_frequency_band_mask(resid_fft.shape, cfg.low_freq_band_cutoff, cfg.high_freq_band_cutoff, device)
            # --- End Mask Selection ---

            # Calculate score: mean weighted magnitude
            score = (resid_fft * mask).sum() / (mask.sum() + 1e-6) # Sum over H, W, C dims
            per_video_scores.append(score.item())
            per_video_labels.append(int(y))

    if len(set(per_video_labels)) < 2:
        print("[WARN] Need both normal and anomalous samples for PR curve.")
        return None # Return None if only one class found

    # Apply temporal smoothing if configured
    scores_final = np.array(per_video_scores)
    if getattr(cfg, "temporal_smoothing", False):
         scores_final = moving_average(scores_final, window=cfg.smoothing_window)
         # Adjust labels if smoothing reduces array size (mode='valid')
         if len(scores_final) < len(per_video_labels):
              pad = cfg.smoothing_window // 2
              per_video_labels = per_video_labels[pad:-pad] if pad > 0 else per_video_labels


    # Calculate Precision-Recall curve and AUC
    precision, recall, _ = precision_recall_curve(per_video_labels, scores_final)
    pr_auc = auc(recall, precision)

    # Plot PR curve
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, lw=2, label=f'Frequency Score PR (AUC={pr_auc:.4f})')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title('Precision-Recall Curve using Adaptive Frequency Score')
    plt.grid(True, linestyle='--', alpha=0.6); plt.legend()
    plt.savefig(os.path.join(out_dir, 'frequency_score_pr_curve.jpg'), dpi=300); plt.close()
    print(f"[INFO] Saved Frequency Score PR curve (AUC: {pr_auc:.4f}).")

    return pr_auc # Return the PR AUC value

def analyze_frequency_domain(model, data_loader, device, out_dir, cfg):
    """Analyzes and plots the average FFT spectrum for normal vs anomalous residuals."""
    print("\n[INFO] Starting FFT Analysis ...")
    model.eval()
    if not os.path.exists(cfg.label_npy):
        print("[WARN] label npy not found; skipping FFT analysis")
        return
    labels = np.load(cfg.label_npy, allow_pickle=True)
    normal_spectrums, anomalous_spectrums = [], []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(data_loader, desc="FFT Analysis")):
            frames = batch['frames']
            inp = frames[:, :cfg.num_frames].to(device)
            tgt = frames[:, cfg.num_frames].to(device)
            try: # Get label
                fn = int(os.path.basename(data_loader.dataset.video_frames[i + cfg.num_frames]).split('.')[0].split('_')[-1])
                y = labels[fn] if fn < len(labels) else 0
            except Exception:
                y = 0

            pred = model(inp)
            # Calculate grayscale residual on CPU
            res_gray = torch.mean(torch.abs(tgt - pred), dim=1).squeeze(0).detach().cpu().numpy()
            # Calculate FFT magnitude spectrum
            f_transform = np.fft.fft2(res_gray)
            f_transform_shifted = np.fft.fftshift(f_transform)
            magnitude_spectrum = 20 * np.log(np.abs(f_transform_shifted) + 1) # Log magnitude

            # Append to respective lists
            (anomalous_spectrums if y == 1 else normal_spectrums).append(magnitude_spectrum)

    if not normal_spectrums or not anomalous_spectrums:
        print("[WARN] Need samples from both normal and anomalous classes for FFT analysis comparison.")
        return

    # Calculate average spectra
    avg_normal = np.mean(normal_spectrums, axis=0)
    avg_anomalous = np.mean(anomalous_spectrums, axis=0)

    # Plot 2D Spectra Comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Frequency Domain Analysis of Residual Maps', fontsize=16)
    im0 = axes[0].imshow(avg_normal, cmap='viridis'); axes[0].set_title('Average Spectrum: Normal')
    fig.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(avg_anomalous, cmap='viridis'); axes[1].set_title('Average Spectrum: Anomalous')
    fig.colorbar(im1, ax=axes[1])
    diff = avg_anomalous - avg_normal
    vmax = np.abs(diff).max()
    im2 = axes[2].imshow(diff, cmap='coolwarm', vmin=-vmax, vmax=vmax); axes[2].set_title('Difference (Anomalous - Normal)')
    fig.colorbar(im2, ax=axes[2])
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    fft_2d_path = os.path.join(out_dir, 'fft_analysis_2D.jpg')
    try:
        plt.savefig(fft_2d_path, dpi=150)
        print(f"[INFO] Saved 2D FFT analysis plot to {fft_2d_path}")
    except Exception as e:
        print(f"[ERROR] Failed to save 2D FFT plot: {e}")
    plt.close()

    # Plot 1D Radial Profile Comparison
    plot_radial_profile(normal_spectrums, anomalous_spectrums, out_dir)
    print("[INFO] FFT Analysis Finished.")
    
def plot_radial_profile(normal_spectrums, anomalous_spectrums, out_dir):
    """Calculates and plots the 1D radially averaged FFT spectrum."""
    def calculate_profiles(spectrums):
        if not spectrums: return None, None
        h, w = spectrums[0].shape
        center = (h // 2, w // 2)
        # Calculate distance of each pixel from the center
        y, x = np.indices((h, w))
        distances = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)
        all_profiles = []
        for s in spectrums:
             # Calculate average magnitude for each distance (radius)
            counts = np.bincount(distances.ravel())
            sums = np.bincount(distances.ravel(), weights=s.ravel())
            # Avoid division by zero for distances with no pixels
            profile = sums / np.where(counts > 0, counts, 1)
            all_profiles.append(profile)
        # Pad profiles to the same length (max radius)
        max_len = max(len(p) for p in all_profiles)
        padded = [np.pad(p, (0, max_len - len(p)), mode='edge') for p in all_profiles]
        # Return mean and std dev across all samples
        return np.mean(padded, axis=0), np.std(padded, axis=0)

    mean_n, std_n = calculate_profiles(normal_spectrums)
    mean_a, std_a = calculate_profiles(anomalous_spectrums)

    if mean_n is None or mean_a is None:
         print("[WARN] Could not calculate radial profiles (missing normal or anomalous samples).")
         return

    plt.figure(figsize=(12, 7))
    x_axis = np.arange(len(mean_n))
    # Plot normal profile with std dev band
    plt.plot(x_axis, mean_n, 'b-', lw=2, label='Normal Residuals (Mean)')
    plt.fill_between(x_axis, mean_n - std_n, mean_n + std_n, color='b', alpha=0.2)
    # Plot anomalous profile with std dev band
    plt.plot(x_axis, mean_a, 'r-', lw=2, label='Anomalous Residuals (Mean)')
    plt.fill_between(x_axis, mean_a - std_a, mean_a + std_a, color='r', alpha=0.2)
    # Use log scale for better visualization
    plt.yscale('log'); plt.xscale('log')
    plt.title('1D Radially Averaged FFT Spectrum (Log-Log Scale)')
    plt.xlabel('Frequency (Distance from center, pixels, log scale)')
    plt.ylabel('Average Log Magnitude (log scale)')
    plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.xlim(left=1) # Start x-axis from 1 for log scale
    radial_path = os.path.join(out_dir, 'fft_radial_profile.jpg')
    try:
        plt.savefig(radial_path, dpi=150)
        print(f"[INFO] Saved 1D radial profile plot to {radial_path}")
    except Exception as e:
        print(f"[ERROR] Failed to save radial profile plot: {e}")
    plt.close()

def analyze_attention_maps(model, data_loader, device, out_dir, cfg, num_samples=5):
    """Generates visualizations of attention maps overlaid on anomalous frames."""
    print("\n[INFO] Starting Attention Map Visualization...")
    model.eval() # Ensure model is in eval mode
    if not os.path.exists(cfg.label_npy):
        print("[WARN] label npy not found; skipping attention visualization.")
        return
    labels = np.load(cfg.label_npy, allow_pickle=True)
    anomalous_found = 0

    # Get the encoder part of the model (handles DataParallel)
    encoder = (model.module.encoder if hasattr(model, 'module') else model.encoder)
    patch_size = encoder.patch_embed.patch_size[0]
    grid_size = cfg.image_size // patch_size

    with torch.no_grad():
        for i, batch in enumerate(tqdm(data_loader, desc="Attention Viz")):
            if anomalous_found >= num_samples:
                break # Stop after finding enough samples

            frames = batch['frames']
            # Get the label for the target frame
            try:
                fn = int(os.path.basename(data_loader.dataset.video_frames[i + cfg.num_frames]).split('.')[0].split('_')[-1])
                y = labels[fn] if fn < len(labels) else 0
            except Exception:
                y = 0

            # Only process anomalous frames
            if y != 1:
                continue

            tgt_frame = frames[:, cfg.num_frames].to(device) # (B, C, H, W), B=1 for validation loader

            # --- Get Attention Map ---
            # Forward pass through the encoder to trigger attention map storage
            _ = (model.module if hasattr(model,'module') else model).forward_features(tgt_frame)

            # Retrieve attention map (handle potential ensemble)
            if getattr(cfg, "use_attention_ensemble", False):
                K = min(getattr(cfg, "attention_ensemble_num", 3), len(encoder.blocks))
                attns_raw = []
                for blk in encoder.blocks[-K:]:
                    a = blk.attn.get_attention_map() # (B, heads, N, N) N=1+num_patches
                    if a is not None:
                        attns_raw.append(a)
                if not attns_raw:
                    attn_raw = encoder.blocks[-1].attn.get_attention_map()
                else:
                    # Stack and average over blocks
                    attn_raw = torch.stack(attns_raw, dim=0).mean(dim=0)
            else:
                 attn_raw = encoder.blocks[-1].attn.get_attention_map()

            if attn_raw is None:
                print(f"[WARN] Could not retrieve attention map for sample {i}.")
                continue

            # Process attention: Average over heads, select CLS token's attention to patches
            # attn_raw shape: (B, heads, N, N), where N = 1 (CLS) + grid_size*grid_size
            cls_attn_to_patches = attn_raw.mean(dim=1)[:, 0, 1:] # (B, N_patches)

            # Reshape to 2D grid
            attn_grid = cls_attn_to_patches.reshape(-1, grid_size, grid_size).unsqueeze(1) # (B, 1, grid, grid)

            # Upsample attention map to image size
            attn_upsampled = F.interpolate(attn_grid,
                                           size=(cfg.image_size, cfg.image_size),
                                           mode='bilinear',
                                           align_corners=False)

            # Normalize map to [0, 1] for visualization
            map_min = attn_upsampled.min()
            map_max = attn_upsampled.max()
            attn_norm = (attn_upsampled - map_min) / (map_max - map_min + 1e-6)

            # --- Create Visualization ---
            # Convert target frame to displayable format [0, 255] uint8 HWC BGR
            img_np = (tgt_frame.squeeze(0).permute(1, 2, 0).detach().cpu().numpy() + 1.0) * 127.5
            img_np = img_np.astype(np.uint8)
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR) # Convert to BGR for OpenCV

            # Create heatmap from normalized attention map
            heatmap_np = (attn_norm.squeeze(0).squeeze(0).cpu().numpy() * 255).astype(np.uint8)
            heatmap_color = cv2.applyColorMap(heatmap_np, cv2.COLORMAP_JET)

            # Overlay heatmap on the original image
            overlaid_img = cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)

            # Concatenate Original | Heatmap | Overlay
            combined_img = np.concatenate((img_bgr, heatmap_color, overlaid_img), axis=1)

            # Save the image
            save_path = os.path.join(out_dir, f"attention_map_anomaly_{anomalous_found}.jpg")
            cv2.imwrite(save_path, combined_img)
            anomalous_found += 1

    print(f"[INFO] Saved {anomalous_found} attention map visualizations.")
# -----------------------------
# SECTION 6: TRAIN / VALID
# -----------------------------

def radial_mask_from_mlp(mlp, shape, device):
    """Generates the frequency mask using the RadialMaskMLP."""
    _, _, H, W = shape
    cy, cx = H // 2, W // 2
    # Create normalized radial distance grid
    yy, xx = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing='ij')
    dist = torch.sqrt((xx - cx).float()**2 + (yy - cy).float()**2)
    radius_max = dist.max() + 1e-8 # Avoid division by zero if dist is all zeros
    rnorm = dist / radius_max
    # Get weights from MLP and ensure correct shape for broadcasting
    mask = mlp(rnorm)
    mask = mask.unsqueeze(0).unsqueeze(0) # -> (1, 1, H, W)
    return mask

def moving_average(a, window=3):
    """Applies a simple moving average filter."""
    if window <= 1 or len(a) < window:
        return np.array(a) # Return original if window is too small or array too short
    pad = window // 2
    # Use 'reflect' padding for better edge handling than 'edge'
    a_p = np.pad(a, (pad, pad), mode='reflect')
    kernel = np.ones(window) / window
    # Use 'valid' mode to ensure output length matches original (after adjusting labels)
    return np.convolve(a_p, kernel, mode='valid')

def train_epoch(epoch, model, loader, optimizer, lr_scheduler, metrics, device, cfg):
    """Runs a single training epoch."""
    model.train() # Set model to training mode
    metrics.reset()
    # Initialize baseline loss functions
    l1 = nn.L1Loss().to(device)
    ssim = SSIM().to(device)
    ploss = PerceptualLoss().to(device)

    for batch in tqdm(loader, desc=f"Training Epoch {epoch}"):
        frames = batch['frames']
        inp = frames[:, :cfg.num_frames].to(device)
        tgt = frames[:, cfg.num_frames].to(device)

        optimizer.zero_grad()
        pred = model(inp)

        # --- Calculate Standard Reconstruction Loss ---
        pred_n, tgt_n = (pred + 1) / 2, (tgt + 1) / 2 # Normalize to [0, 1] for SSIM/Perceptual
        main_loss = cfg.l1_weight * l1(pred, tgt) \
                   + cfg.ssim_weight * (1 - ssim(pred_n, tgt_n)) \
                   + cfg.perceptual_weight * ploss(pred_n, tgt_n)

        # --- Add Frequency Guidance for MLP (if enabled) ---
        total_loss = main_loss
        if cfg.use_learnable_mask and hasattr(model, "radial_mlp") and model.radial_mlp is not None:
             # Calculate residual FFT magnitude (no gradients needed for this part)
             with torch.no_grad():
                 residual = torch.abs(tgt - pred)
                 residual_gray = torch.mean(residual, dim=1, keepdim=True) # Use grayscale
                 resid_fft_mag = torch.abs(torch.fft.fftshift(torch.fft.fft2(residual_gray, norm='ortho')))

             # Get learned mask from the MLP (part of computation graph)
             learned_mask = radial_mask_from_mlp(model.radial_mlp, resid_fft_mag.shape, device)

             # Simple L1 loss in frequency domain using learned mask
             # Penalize non-zero values in the weighted spectrum
             freq_loss = F.l1_loss(resid_fft_mag * learned_mask, torch.zeros_like(resid_fft_mag))
             # Combine losses
             total_loss = main_loss + cfg.frequency_guidance_weight * freq_loss

        # --- Backpropagation ---
        total_loss.backward()
        optimizer.step()
        # Step the learning rate scheduler
        if lr_scheduler is not None:
            lr_scheduler.step()

        metrics.update('loss', float(total_loss.item())) # Log combined loss

    print(f"[TRAIN] Epoch {epoch:03d} Avg Loss: {metrics.result()['loss']:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    return metrics.result()

def valid_epoch(epoch, model, loader, metrics, device, frame_save_dir, cfg):
    """Runs a single validation epoch."""
    model.eval() # Set model to evaluation mode
    metrics.reset()
    # Store scores for both methods and labels
    all_scores_learned_freq, all_scores_baseline, all_labels = [], [], []

    # Load ground truth labels
    if not os.path.exists(cfg.label_npy):
        print("[WARN] Validation labels not found. Cannot calculate AUC/EER.")
        return {'loss': -1, 'auc': 0.0, 'eer': 1.0, 'fpr': np.array([0, 1]), 'tpr': np.array([0, 1]), 'auc_baseline': 0.0}
    labels = np.load(cfg.label_npy, allow_pickle=True)

    with torch.no_grad(): # Disable gradient calculations for validation
        for i, batch in enumerate(tqdm(loader, desc=f"Validating Epoch {epoch}")):
            frames = batch['frames']
            inp = frames[:, :cfg.num_frames].to(device)
            tgt = frames[:, cfg.num_frames].to(device)
            pred = model(inp)

            # --- 1. Baseline Anomaly Score (L1 + SSIM) ---
            # Use a separate function for clarity if needed, or inline
            score_baseline_batch = compute_combined_score(tgt, pred, use_temporal_smoothing=False) # No smoothing here, applied later if needed
            all_scores_baseline.extend(score_baseline_batch.tolist())

            # --- 2. Learned Frequency Score ---
            residual = torch.abs(tgt - pred)
            residual_gray = torch.mean(residual, dim=1, keepdim=True)
            resid_fft_mag = torch.abs(torch.fft.fftshift(torch.fft.fft2(residual_gray, norm='ortho')))

            # Get learned or fixed mask
            if cfg.use_learnable_mask and hasattr(model, "radial_mlp") and model.radial_mlp is not None:
                learned_mask = radial_mask_from_mlp(model.radial_mlp, resid_fft_mag.shape, device)
            else: # Fallback to fixed mask
                learned_mask = create_frequency_band_mask(resid_fft_mag.shape, cfg.low_freq_band_cutoff, cfg.high_freq_band_cutoff, device)

            # Calculate score: mean weighted magnitude per sample in batch
            score_learned_freq_batch = torch.mean((resid_fft_mag * learned_mask).view(tgt.size(0), -1), dim=1)
            all_scores_learned_freq.extend(score_learned_freq_batch.detach().cpu().numpy().tolist())

            # Get corresponding label for each sample in the batch (assuming batch size 1 here)
            try:
                fn = int(os.path.basename(loader.dataset.video_frames[i + cfg.num_frames]).split('.')[0].split('_')[-1])
                label = int(labels[fn]) if fn < len(labels) else 0
                all_labels.append(label)
            except Exception as e:
                # print(f"Label extraction warning: {e}")
                all_labels.append(0) # Default to normal if error

            # Save visualization for the first few frames
            if i < 5:
                save_anomaly_visualization(tgt[0:1], pred[0:1], epoch, i, frame_save_dir)

    # --- Post-process scores (Temporal Smoothing) ---
    scores_baseline_final = np.array(all_scores_baseline)
    scores_learned_final = np.array(all_scores_learned_freq)
    labels_final = np.array(all_labels[:len(scores_baseline_final)]) # Ensure labels match score length

    if getattr(cfg, "temporal_smoothing", False):
         scores_baseline_final = moving_average(scores_baseline_final, window=cfg.smoothing_window)
         scores_learned_final = moving_average(scores_learned_final, window=cfg.smoothing_window)
         # Adjust labels if smoothing reduces array size
         if len(scores_learned_final) < len(labels_final):
              pad = cfg.smoothing_window // 2
              labels_final = labels_final[pad:-pad] if pad > 0 else labels_final


    # --- Calculate Metrics ---
    results = {}
    if len(set(labels_final)) < 2 or len(labels_final) == 0:
        print("[WARN] Not enough samples or only one class found after processing. Setting metrics to default.")
        results = {'auc': 0.0, 'eer': 1.0, 'fpr': np.array([0, 1]), 'tpr': np.array([0, 1]), 'auc_baseline': 0.0}
    else:
        # Calculate AUC for both methods
        results['auc_learned'] = roc_auc_score(labels_final, scores_learned_final)
        results['auc_baseline'] = roc_auc_score(labels_final, scores_baseline_final)

        # Calculate EER and ROC curve using the primary (learned frequency) score
        fpr, tpr, _ = roc_curve(labels_final, scores_learned_final)
        fnr = 1 - tpr
        eer_idx = np.nanargmin(np.abs(fnr - fpr))
        eer = fpr[eer_idx]
        results['eer'] = float(eer)
        results['fpr'] = fpr
        results['tpr'] = tpr

        print(f"[VAL]   Epoch {epoch:03d} AUCs | Learned Freq: {results['auc_learned']:.4f} | Baseline: {results['auc_baseline']:.4f} | EER (Learned): {results['eer']:.4f}")

    # Update metrics tracker with primary score results
    metrics.update('loss', float(np.mean(scores_learned_final)) if len(scores_learned_final)>0 else -1) # Log mean score as 'loss'
    metrics.update('auc', results.get('auc_learned', 0.0))
    metrics.update('eer', results.get('eer', 1.0))

    # Return dictionary including both AUCs and ROC data for primary score
    return {
        'loss': metrics.result()['loss'],
        'auc': results.get('auc_learned', 0.0), # Primary AUC
        'eer': results.get('eer', 1.0),
        'auc_baseline': results.get('auc_baseline', 0.0), # Baseline for comparison
        'fpr': results.get('fpr', np.array([0, 1])),
        'tpr': results.get('tpr', np.array([0, 1]))
    }

def compute_combined_score(tgt, pred, use_temporal_smoothing=False, smoothing_window=3):
    """Computes the baseline L1+SSIM score."""
    l1_res = torch.mean(torch.abs(tgt - pred).view(tgt.size(0), -1), dim=1)
    ssim_mod = SSIM(size_average=False).to(pred.device) # Ensure SSIM is on correct device
    # Ensure inputs to SSIM are on the same device and in [0,1] range
    pred_norm = ((pred + 1) / 2).to(pred.device)
    tgt_norm = ((tgt + 1) / 2).to(pred.device)
    ssim_res = (1 - ssim_mod(pred_norm, tgt_norm))

    score = 0.5 * l1_res + 0.5 * ssim_res
    score_np = score.detach().cpu().numpy()

    if use_temporal_smoothing:
        score_np = moving_average(score_np, window=smoothing_window)

    return score_np


# -----------------------------
# SECTION 7: MAIN EXECUTION BLOCK
# -----------------------------

def main():
    cfg = get_config()
    os.makedirs(cfg.out_dir, exist_ok=True)
    frame_dir = os.path.join(cfg.out_dir, "output_frames")
    ckpt_dir = os.path.join(cfg.out_dir, "checkpoints")
    os.makedirs(frame_dir, exist_ok=True)
    os.makedirs(ckpt_dir, exist_ok=True)

    # Save config json for reproducibility
    config_dict = {k: getattr(cfg, k) for k in dir(cfg) if not k.startswith("_") and not callable(getattr(cfg, k))}
    with open(os.path.join(cfg.out_dir, "run_config.json"), "w") as f:
        json.dump(config_dict, f, indent=2)
    print(f"[INFO] Run Configuration saved to {os.path.join(cfg.out_dir, 'run_config.json')}")

    device, device_ids = setup_device(cfg.n_gpu)

    # Basic transform (numpy HWC -> tensor CHW) for non-augmented data
    # Note: Normalization to [-1, 1] happens in np_load_frame
    data_transform = transforms.Compose([transforms.ToTensor()])

    print(f"[INFO] Setting up datasets...")
    train_set = AnomalyDataset(cfg.train_folder, data_transform, cfg.image_size, cfg.image_size, cfg.num_frames, augment=True)
    test_set = AnomalyDataset(cfg.test_folder, data_transform, cfg.image_size, cfg.image_size, cfg.num_frames, augment=False)

    if len(train_set) == 0:
        print("[ERROR] Training dataset is empty. Check train_folder path and contents.")
        return
    if len(test_set) == 0:
        print("[ERROR] Test dataset is empty. Check test_folder path and contents.")
        return

    train_loader = data.DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, drop_last=True, pin_memory=True)
    # Use batch_size=1 for validation as scoring is often per-frame
    test_loader = data.DataLoader(test_set, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    print(f"[INFO] Train batches: {len(train_loader)}, Test samples: {len(test_loader)}")

    # Initialize model, passing config for MLP creation
    print(f"[INFO] Initializing model...")
    model = VisionTransformerSOTA(cfg.image_size, cfg.num_frames, cfg.unfreeze_layers, cfg=cfg).to(device)
    if len(device_ids) > 1:
        model = nn.DataParallel(model, device_ids=device_ids)

    # Optimizer (include MLP params if they exist and require grad)
    params_to_optimize = list(filter(lambda p: p.requires_grad, model.parameters()))
    print(f"[INFO] Optimizing {len(params_to_optimize)} parameter groups.")
    optimizer = torch.optim.AdamW(params_to_optimize, lr=cfg.lr, weight_decay=cfg.wd)

    # Learning rate schedulers (Warmup + Cosine Annealing)
    total_steps = max(1, len(train_loader) * cfg.epochs)
    warmup_iters = max(1, cfg.warmup_steps)
    main_iters = max(1, total_steps - warmup_iters)

    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_iters)
    main_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_iters, eta_min=cfg.eta_min)
    lr_sched = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, main_sched], milestones=[warmup_iters])
    print(f"[INFO] LR Scheduler: Warmup ({warmup_iters} steps) -> Cosine ({main_iters} steps)")

    # Metric trackers
    train_metrics = MetricTracker('loss')
    # Track both AUCs during validation
    valid_metrics = MetricTracker('loss', 'auc', 'eer', 'auc_baseline')

    # Setup logging CSV
    log_csv = os.path.join(cfg.out_dir, 'training_log.csv')
    with open(log_csv, 'w') as f:
        f.write('epoch,train_loss,val_auc_learned,val_auc_baseline,val_eer,val_pr_auc\n')
    print(f"[INFO] Logging results to {log_csv}")

    best_auc = -1.0
    best_pr_auc = -1.0
    roc_hist = []
    best_epoch_data = None

    if cfg.train:
        print(f"[INFO] Starting training for {cfg.epochs} epochs...")
        for epoch in range(1, cfg.epochs + 1):
            start_time = time.time()
            tr = train_epoch(epoch, model, train_loader, optimizer, lr_sched, train_metrics, device, cfg)
            vr = valid_epoch(epoch, model, test_loader, valid_metrics, device, frame_dir, cfg)

            # Evaluate PR AUC for frequency score separately after each validation epoch
            pr_auc = None
            if cfg.do_freq_score_pr:
                 # Ensure model is in eval mode for this function too
                 model.eval()
                 pr_auc = evaluate_frequency_score(model, test_loader, device, cfg.out_dir, cfg)
                 pr_auc = pr_auc if pr_auc is not None else 0.0
            else:
                 pr_auc = 0.0 # Placeholder if not evaluated

            # Store ROC history using primary AUC (learned frequency score)
            roc_hist.append({'epoch': epoch, **{k: vr[k] for k in ('auc', 'fpr', 'tpr') if k in vr}})

            # Log results to CSV
            with open(log_csv, 'a') as f:
                # Log learned AUC, baseline AUC, EER, and PR AUC
                f.write(f"{epoch},{tr['loss']:.6f},{vr['auc']:.6f},{vr['auc_baseline']:.6f},{vr['eer']:.6f},{pr_auc:.6f}\n")

            # Checkpoint saving (based on primary AUC - learned frequency score)
            current_auc = vr['auc']
            state = {
                'epoch': epoch,
                # Save state_dict correctly for DataParallel
                'state_dict': model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': lr_sched.state_dict()
            }
            torch.save(state, os.path.join(ckpt_dir, 'current.pth'))

            if current_auc > best_auc:
                best_auc = current_auc
                best_pr_auc = pr_auc # Store corresponding PR AUC
                # Update best_epoch_data with all relevant metrics from vr
                best_epoch_data = {'epoch': epoch, **vr, 'pr_auc': best_pr_auc}
                torch.save(state, os.path.join(ckpt_dir, 'best.pth'))
                print(f"[CKPT] New best AUC (Learned Freq) {best_auc:.4f} at epoch {epoch}. Saved best model.")

            epoch_time = time.time() - start_time
            print(f"[INFO] Epoch {epoch}/{cfg.epochs} finished in {epoch_time:.2f}s")


        print("\n[INFO] Training finished.")
        save_roc_curves(roc_hist, best_epoch_data, cfg.out_dir)

        # Reload best model for final analyses
        best_path = os.path.join(ckpt_dir, 'best.pth')
        if os.path.exists(best_path) and best_epoch_data:
            print(f"[INFO] Loading best model from epoch {best_epoch_data['epoch']} for final analyses.")
            # Re-initialize model architecture
            analysis_model = VisionTransformerSOTA(cfg.image_size, cfg.num_frames, cfg.unfreeze_layers, cfg=cfg).to(device)
            chk = torch.load(best_path, map_location=device)

            # Handle DataParallel state_dict keys
            state_dict = chk['state_dict']
            if list(state_dict.keys())[0].startswith('module.'):
                 new_state_dict = OrderedDict([(k[7:], v) for k, v in state_dict.items()])
                 analysis_model.load_state_dict(new_state_dict)
            else:
                 analysis_model.load_state_dict(state_dict)
            analysis_model.eval() # Ensure model is in eval mode

            # Run final analyses using the loaded best model
            if cfg.do_fft_analysis:
                analyze_frequency_domain(analysis_model, test_loader, device, cfg.out_dir, cfg)
            if cfg.do_attention_viz:
                analyze_attention_maps(analysis_model, test_loader, device, cfg.out_dir, cfg)
            # Re-run frequency score PR evaluation on best model if not already done or for confirmation
            if cfg.do_freq_score_pr and best_pr_auc < 0: # Re-run if not captured during training loop
                 evaluate_frequency_score(analysis_model, test_loader, device, cfg.out_dir, cfg)

        else:
             print("[WARN] Best model checkpoint not found or no validation performed. Skipping final analyses.")

    # Final summary JSON
    summary = {
        "best_epoch": best_epoch_data['epoch'] if best_epoch_data else None,
        "best_val_auc_learned_freq": float(best_auc) if best_epoch_data else None,
        "best_val_auc_baseline": float(best_epoch_data['auc_baseline']) if best_epoch_data else None,
        "best_val_eer": float(best_epoch_data['eer']) if best_epoch_data else None,
        "best_val_pr_auc": float(best_pr_auc) if best_epoch_data else None,
        "config": config_dict
    }
    summary_path = os.path.join(cfg.out_dir, "results_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[DONE] Results summary saved to: {summary_path}")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

[INFO] Run Configuration saved to /kaggle/working/run_config.json
[INFO] Using device: cuda:0
[INFO] Setting up datasets...
[INFO] Train batches: 110, Test samples: 2349
[INFO] Initializing model...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[INFO] Freezing 92 / 152 parameter groups in the encoder.
[INFO] Using Learnable Radial Mask MLP.
[INFO] Optimizing 84 parameter groups.
[INFO] LR Scheduler: Warmup (100 steps) -> Cosine (2650 steps)
[INFO] Logging results to /kaggle/working/training_log.csv
[INFO] Starting training for 25 epochs...


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 229MB/s]  
Training Epoch 1:   0%|          | 0/110 [00:00<?, ?it/s]/tmp/ipykernel_37/3743554632.py:216: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/

...

[INFO] Saved Frequency Score PR curve (AUC: 0.1283).
[INFO] Epoch 25/25 finished in 398.27s

[INFO] Training finished.
[INFO] Saved all epochs ROC curve to /kaggle/working/all_epochs_roc_curve.jpg
[INFO] Saved best epoch ROC curve to /kaggle/working/publication_style_roc_curve.jpg
[INFO] Loading best model from epoch 7 for final analyses.
[INFO] Freezing 92 / 152 parameter groups in the encoder.
[INFO] Using Learnable Radial Mask MLP.

[INFO] Starting FFT Analysis ...


FFT Analysis: 100%|██████████| 2349/2349 [02:28<00:00, 15.87it/s]


[INFO] Saved 2D FFT analysis plot to /kaggle/working/fft_analysis_2D.jpg
[INFO] Saved 1D radial profile plot to /kaggle/working/fft_radial_profile.jpg
[INFO] FFT Analysis Finished.

[INFO] Starting Attention Map Visualization...


Attention Viz:   1%|▏         | 31/2349 [00:01<02:13, 17.35it/s]

[INFO] Saved 5 attention map visualizations.
[DONE] Results summary saved to: /kaggle/working/results_summary.json
